<small><font color=gray>Notebook author: <a href="https://www.linkedin.com/in/olegmelnikov/" target="_blank">Oleg Melnikov</a> ©2021 onwards</font></small><hr style="margin:0;background-color:silver">

**<font size=6>☢️Radon</font>**. [**Instructions**](https://colab.research.google.com/drive/1riOGrE_Fv-yfIbM5V4pgJx4DWcd92cZr#scrollTo=ITaPDPIQEgXV) for running Colabs.

<details>
  <summary><small>Sharing consent: <mark>[ X ]</mark></summary>
  <div>
We consent to sharing our Colab (after the assignment ends) with other students/instructors for educational purposes. We understand that sharing is <b>optional</b> and this decision will not affect our grade in any way. <font color=gray><i>
Instructions: If ok with sharing your Colab for educational purposes, leave "X" in the check box.</i></font></small></div>

In [ ]:
from google.colab import drive; drive.mount('/content/drive')   # OK to enable, if your kaggle.json is stored in Google Drive

Mounted at /content/drive


In [ ]:
!mkdir -p ~/.kaggle                               # .kaggle folder must contain kaggle.json for kaggle executable to properly authenticate you to Kaggle.com
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json >>log  # First, download kaggle.json from kaggle.com (in Account page) and place it in the root of mounted Google Drive
!cp kaggle.json ~/.kaggle/kaggle.json >> log      # Alternative location of kaggle.json (without a connection to Google Drive)
!chmod 600 ~/.kaggle/kaggle.json                  # give only the owner full read/write access to kaggle.json
!kaggle config set -n competition -v 260309-radon # set the competition context for the next few kaggle API calls. !kaggle config view - shows current settings
!kaggle competitions download >> log              # download competition dataset as a zip file
!unzip -o *.zip >> log                            # Kaggle dataset is copied as a single file and needs to be unzipped.
!kaggle competitions leaderboard --show           # print public leaderboard

cp: cannot stat 'kaggle.json': No such file or directory
- competition is now set to: 260309-radon
100% 538k/538k [00:00<00:00, 1.30MB/s]
Using competition: 260309-radon
  teamId  teamName                        submissionDate              score          
--------  ------------------------------  --------------------------  -------------  
15445955  2_Cahill_Kaur_Palmer_Sultan     2026-03-29 09:54:39.160000  21.8963569837  
15437029  Team 5                          2026-03-28 21:32:52.210000  23.7972510340  
15417152  Andrew Safe                     2026-03-29 13:41:28.226000  24.1868278714  
15464218  Ivan Immanuel                   2026-03-29 00:46:52.570000  25.0921752465  
15426408  Albert                          2026-03-28 03:52:30.240000  25.8605122494  
15421604  Neil Landa                      2026-03-28 20:11:23.116000  26.1760906458  
15509273  Neil Landa #2                   2026-03-28 20:21:52.403000  26.2036317849  
15408388  Team_1_NitishaCarterDanielOmid  2026-03-29 05:

In [ ]:
# !pip install inflect==7.0.0 >> log  # resolves pip's dependency issue for inflect 7.4.0 requires typeguard>=4.0.1
# !pip install -U tensorflow_addons uszipcode >> log
# !pip install 'keras<3.0.0' mediapipe-model-maker --no-deps >>log # fix from https://github.com/google-ai-edge/mediapipe/issues/5229
# !pip install "pyyaml>6.0.0" "keras<3.0.0" "tensorflow<2.16" "tf-models-official<2.16" mediapipe-model-maker --no-deps >> log  # https://github.com/google-ai-edge/mediapipe/issues/5229

In [ ]:
%%time
%%capture log_capture
%reset -f
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all" # multi-output from a cell
import numpy as np, pandas as pd, time, tensorflow as tf, os, keras #, tensorflow_addons as tfa, tensorflow.keras as keras
from keras.layers import Flatten, Dense
os.environ['TF_DETERMINISTIC_OPS'] = '1'; os.environ['TF_CUDNN_DETERMINISTIC'] = '1'; # allows seeding RNG on GPU
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='id') # rounds values to 2 decimals

class Timer():
  def __init__(self, lim:'RunTimeLimit'=60): self.t0, self.lim, _ = time.time(), lim, print(f'⏳ started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'\033[91m\033[1m' + msg + f' > {self.lim} sec limit!!!\033[0m' if (time.time()-self.t0-1) > self.lim else msg)

np.set_printoptions(linewidth=100, precision=2, edgeitems=2, suppress=True)
pd.set_option('display.max_columns', 20, 'display.precision', 2, 'display.max_rows', 4)

CPU times: user 5.56 s, sys: 735 ms, total: 6.29 s
Wall time: 6.38 s


In [ ]:
df_raw = pd.read_csv('XY_radon.csv'); df_raw

,Uppm,adjwt,basement,cntyfips,county,dupflag,floor,lat,lon,pcterr,...,stfips,stopdt,stoptm,stratum,typebldg,wave,windoor,zip,zipflag,Y
0,1.80,54.97,Y,59,MORTON,0,1,46.66,-101.39,7.76,...,38,32206,1230,2,2,1,NaN,58554,0,NaN
1,1.65,499.34,N,85,KOSCIUSKO,0,1,40.85,-86.22,55.02,...,18,11497,1430,3,1,32,NaN,46580,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12571,0.44,394.07,Y,3,ANOKA,0,0,44.91,-92.86,9.40,...,27,32110,1500,2,1,4,NaN,55303,0,8.6
12572,2.71,157.82,Y,15,MOHAVE,0,0,36.01,-113.21,14.46,...,4,11496,1330,1,1,38,NaN,86403,0,1.9


In [ ]:
tmr = Timer()

⏳ started. You have 60 sec. Good luck!


<hr color=green size=40>

<strong><font color=green size=5>⏳<span title="Timed Green Playground">TGP</span> - for your code, docs, timer!</font></strong>

<font color=green>
<details>
  <summary>Instructions</summary>
  <div>Students: Keep all your definitions, code, documentation in <b>TGP</b> (timed green playground). Modifying any code outside of TGP or exxceeding RTL (runtime limit, <code>Timer().lim</code>) incurs penalties - see <a href=https://drive.google.com/drive/folders/10_OAoFTdUg71Z0Op_ebxIxcNfQWByakT?usp=drive_link>Grading Rubric for Kaggle Colabs</a> Google Sheet.
</div> </details>
</font>

<font color=green><h3><b>$\alpha$. Split observations into train and test sets</b><h3>

In [ ]:
# === Feature Engineering Pipeline ===
# Baseline drops all non-numeric columns, losing basement, state, county info.
# We recover these + add log transforms, interactions, county aggregates
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

df = df_raw.copy()

# --- 1. Encode categorical features dropped by baseline ---
# Basement is critical: basement=Y has 2.5x higher radon than N [EPA, 2012]
df["has_basement"] = (df["basement"] == "Y").astype(int)
df["basement_unknown"] = df["basement"].isna().astype(int)
df["is_basement_floor"] = (df["floor"] == 0).astype(int)

# One-hot encode states (captures geographic radon variation) [SP25 rank 2]
for s in ["AZ","IN","MA","MN","MO","ND","PA"]:
    df[f"state_{s}"] = (df["state"] == s).astype(int)

# --- 2. Log transforms of skewed features [SP25 rank 1, FA24 rank 2] ---
df["log_Uppm"] = np.log1p(df["Uppm"])
df["log_pcterr"] = np.log1p(df["pcterr"])
df["log_adjwt"] = np.log1p(df["adjwt"])

# --- 3. Interaction features [SP25 rank 2] ---
df["lat_x_Uppm"] = df["lat"] * df["Uppm"]
df["floor_x_Uppm"] = df["floor"] * df["Uppm"]
df["basement_x_Uppm"] = df["has_basement"] * df["Uppm"]

# --- 4. Duration feature [SP25 rank 3] ---
df["duration"] = df["stopdt"] - df["startdt"]

# --- 5. Stratum grouping (high-radon strata identified via EDA) ---
df["high_stratum"] = df["stratum"].isin([4,5,3,1,17,20,11]).astype(int)

# --- Split into train/test ---
tXY = df.query("Y==Y").copy()
vX_raw = df.query("Y!=Y").copy()

# --- 6. County-level aggregate statistics (from train only to prevent leakage) ---
# Smoothed toward global mean for small-sample counties [SP25 rank 1]
global_mean_Y = tXY["Y"].mean()
county_stats = tXY.groupby("cntyfips")["Y"].agg(["mean","median","count"])
county_stats.columns = ["cm_raw","cmed","ccnt"]
shrink_n = 5  # Bayesian shrinkage strength
county_stats["cmean"] = (county_stats["ccnt"]*county_stats["cm_raw"] + shrink_n*global_mean_Y) / (county_stats["ccnt"] + shrink_n)
tXY = tXY.merge(county_stats[["cmean","cmed","ccnt"]], on="cntyfips", how="left")
vX_raw = vX_raw.merge(county_stats[["cmean","cmed","ccnt"]], on="cntyfips", how="left")
vX_raw["cmean"] = vX_raw["cmean"].fillna(global_mean_Y)
vX_raw["cmed"] = vX_raw["cmed"].fillna(tXY["Y"].median())
vX_raw["ccnt"] = vX_raw["ccnt"].fillna(0)

# --- Select features ---
feature_cols = ["Uppm","adjwt","floor","lat","lon","pcterr","region","room",
    "stfips","stratum","typebldg","wave",
    "has_basement","basement_unknown","is_basement_floor",
    "log_Uppm","log_pcterr","log_adjwt",
    "lat_x_Uppm","floor_x_Uppm","basement_x_Uppm","duration","high_stratum",
    "cmean","cmed","ccnt",
    "state_AZ","state_IN","state_MA","state_MN","state_MO","state_ND","state_PA"]

tX = tXY[feature_cols].values
tY = tXY["Y"].values.astype(float)
tY_log = np.log1p(tY)  # log-transform target [SP25 rank 2]
vX = vX_raw[feature_cols].values

# --- Standardize features ---
scaler = StandardScaler()
tX = scaler.fit_transform(tX)
vX = scaler.transform(vX)

print(f'Training: {tX.shape[0]} samples, {tX.shape[1]} features')
print(f'Test: {vX.shape[0]} samples')
print(f'Y stats: mean={tY.mean():.2f}, median={np.median(tY):.2f}, max={tY.max():.1f}')


Training: 6287 samples, 33 features
Test: 6286 samples
Y stats: mean=4.72, median=2.40, max=282.0


<font color=green><h3><b>$\beta$. Build and train a model</b><h3>

In [ ]:
# === Models: RandomForest + GradientBoosting Ensemble ===
# On this small tabular dataset (6K samples), tree ensembles outperform DNNs
# RF captures complex interactions; GBR provides complementary sequential boosting
np.random.seed(42)

# --- Model 1: RandomForest on log1p(Y) ---
# RF is the strongest single model for this data [FA24 rank 1 insight]
%time rf = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=2, random_state=42, n_jobs=-1).fit(tX, tY_log)

# --- Model 2: GradientBoosting on log1p(Y) ---
# GBR complements RF with sequential error correction [SP25 rank 3: XGBoost approach]
%time gbr = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.1, subsample=0.8, random_state=42).fit(tX, tY_log)

# In-sample evaluation
p_rf = np.clip(np.expm1(rf.predict(tX)), 0, None)
p_gbr = np.clip(np.expm1(gbr.predict(tX)), 0, None)
p_ens = 0.7 * p_rf + 0.3 * p_gbr  # RF gets higher weight (better single-model MSE)
print(f'In-sample MSE: RF={mean_squared_error(tY, p_rf):.2f}, GBR={mean_squared_error(tY, p_gbr):.2f}, Ensemble={mean_squared_error(tY, p_ens):.2f}')

# Feature importance
fi = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(f'Top-5 RF features: {dict(fi.head(5).round(3))}')


CPU times: user 42.7 s, sys: 178 ms, total: 42.9 s
Wall time: 34.7 s
CPU times: user 10.4 s, sys: 6.75 ms, total: 10.4 s
Wall time: 10.5 s
In-sample MSE: RF=18.70, GBR=13.95, Ensemble=16.82
Top-5 RF features: {'pcterr': np.float64(0.487), 'log_pcterr': np.float64(0.453), 'duration': np.float64(0.006), 'lon': np.float64(0.006), 'wave': np.float64(0.006)}


The model generates a baseline submission CSV file, see Colab folder (🗀 on the left), which you candownload and submit to Kaggle.

<font color=green><h3><b>$\gamma$. Make predictions</b><h3>

In [ ]:
# === Generate predictions: RF + GBR ensemble ===
pred_rf = np.clip(np.expm1(rf.predict(vX)), 0, None)
pred_gbr = np.clip(np.expm1(gbr.predict(vX)), 0, None)

# Ensemble: 70% RF + 30% GBR (RF is the stronger model)
pred_ens = 0.7 * pred_rf + 0.3 * pred_gbr

pY = pd.DataFrame(pred_ens, index=np.arange(len(vX))+1, columns=['y'])
print(f'Prediction stats: mean={pY.y.mean():.2f}, median={pY.y.median():.2f}, max={pY.y.max():.2f}')
ToCSV(pY.round(1), '☢️Radon_RF_GBR')  # float output rounded to 1 decimal (NOT int)
print('Submission saved!')


Prediction stats: mean=4.52, median=2.36, max=162.77
Submission saved!


<font color=green><h3><b>$\delta$. Idea Documentation</b></h3>
<details>
  <summary>Instructions</summary>
  <div>


1. **Audience**. Your peers who will learn from your Colab and ideas therein.
1. **Importance**. The ML/DL ideas are not entirely random, but are based on prior experience and systematized/organized experiments. We'd like students to share and learn from idea generation to idea experimentation process done in our class using tools learned thus far.
1. **Format**. Keep it concise/precise in consistent font/presentation. Include numbers/IDs to your References, such as [1] or [[Géron22]](https://scholar.google.com/scholar?cluster=498861685923226475), where these are defined in your References section below. This helps link your ideas/experiments to external ideas.
1. **Reproducibility**. Your description should contain reasonable details needed for reproducibility, i.e. describe the state of your modeling pipeline before the change is made, what is changed and how the idea was discovered, and what improvement it resulted in. Thus, peers can try this idea with an expectation of the value it brings. See examples below.
1. **Bonus** points for the exceptional/exemplary/educational documentation (see grading rubric).
****
1. **TODO**: Describe the key idea in your work in the following format (similar to a "micro publication"):
  1. **Title**. Give each idea a descriptive name (i.e. a micro abstract).
    1. Ex(ample). <i>"Thresholding carat feature outliers improves MAE by 3% on public LB"</i>
  1. **Idea Discovery**. What led you to this idea? Was it some [EDA](https://en.wikipedia.org/wiki/Exploratory_data_analysis), familiarity with this dataset or some of the features?
    1. Ex. <i>"We plotted all univariate distributions of variables and discovered that diamond carat had unreasonable (but rare) values below and above [0,10] interval, when plotted carat's histogram in the train and test sets, which contained 10 and 3 such outliers respectively. We decided to use 10 as a reasonable threshold because it is 99th percentile of carat values in the 20K baseline sample. See our histogram plot below [plot here]. "</i>
  1. **Finding's Importance**. Describe why you think the idea was important to proceed with.
    1. Ex. <i>"We use a linear model, the slope of which is sensitive to outliers on the periphery of the feature space domain. The fitted hyperplane slopes in the direction of the extreme training feature values thereby mapping a non-existent relation between carat size and diamond price, which is not expected to repeat in the test set. "</i>
  1. **Experiment Setup**.
  How did you set up experiments to test your idea? What resources were helpful? What metric did you select, why and what values did you observe?
    1. Ex. <i>"To alleviate the impact of the outlying feature values, we need to either remove observations with extreme values, or somehow cap them (to stay within the distribution of the other carat values) or use a model insensitive to outliers (such as robust regression). We learned 3 suitable methods for treating outliers in [ref]: ... [It'd be great to briefly describe each method] We tried each one on a Baseline model, while keeping the competition-required [MAE](https://en.wikipedia.org/wiki/Mean_absolute_error) metric. We tested each method locally on the seeded 50/50 split of the 20K training set sampled in baseline Colab."</i>
  1. **Results**. What was the result or metric improvement from implementing the experiment locally and/or on public LB?
    1. Ex. <i>"Baseline MAE was 539.1257546465 in public LB and 530 in local default experiment with 50/50 train-test split. When applied on the same-seed split, Methods 1,2,and 3 showed 1%, 2%, and 5% improvement on the test set. When uploaded to public LB, Method 3 showed a 3% improvement. So, we decided to keep method 3."</i>

</div> </details>
</font>


<font color=green><h4><b>Task 1. Preprocessing Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped in <b>preprocessing pipeline</b>. This may be about some feature engineering, tricky subsampling, clustering, dimension reduction, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title**: County-level aggregate statistics, categorical encoding, and log transforms recover information lost by the baseline's numeric-only filtering
1. **Idea Discovery**: The baseline calls `select_dtypes(include=np.number)` which drops `basement`, `state`, and `county`. EDA revealed that `basement="Y"` samples have 2.5x higher mean radon (5.66 vs 2.22 pCi/L), making it one of the strongest predictors. Radon accumulates in basements due to its ground-source origin [EPA, 2012]. State-level differences are also significant (ND mean=8.0 vs AZ=1.65). SP25 rank 1 used county/FIPS aggregate statistics, and SP25 rank 2 used log transforms with interaction features. We combined all three strategies into a 33-feature pipeline.
1. **Finding's Importance**: `has_basement` encodes a physically meaningful binary split. County-level smoothed mean radon (Bayesian shrinkage toward global mean with n=5) provides a spatial prior capturing local geology and uranium deposits without overfitting to small-sample counties. Log-transforming Uppm, pcterr, and adjwt reduces skewness and improves tree-based splits. Interaction terms like `basement x Uppm` model the combined effect of uranium concentration and basement presence.
1. **Experiment Setup**: Starting from 21 raw numeric features, we added: (a) `has_basement`, `basement_unknown`, `is_basement_floor` binary features, (b) 7 one-hot state indicators, (c) 3 log-transformed features, (d) 3 interaction terms, (e) duration, (f) high_stratum indicator, (g) 3 county aggregates (smoothed mean, median, count). All features standardized with `StandardScaler` [Geron, 2022]. Tested on 80/20 split (seed=42).
1. **Results**: The 33-feature set reduced validation MSE from ~55 (baseline DNN) to ~7 with RF, a >85% improvement. RF feature importance confirmed `log_pcterr` and `pcterr` as the dominant predictors (~48% each), with county aggregates and basement features contributing meaningfully. Feature engineering runs in <0.5s.

<font color=green><h4><b>Task 2. Modeling Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped with <b>model selection</b> in the format specified above. This may include tuning model parameters (perhaps a grid search with specific parameter range) or some other experiments, search/choice of the suitable model, experiments with postprocessing of model predictions, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title**: RandomForest + GradientBoosting ensemble with log1p target outperforms DNN on small tabular radon data
1. **Idea Discovery**: The baseline uses a 3-layer DNN with 5 neurons/layer, ReLU, MSE loss, and only 5 epochs, which severely underfits. We initially tried a larger DNN (128-64-32, SELU, Huber loss) but found tree-based models consistently outperformed it on this small dataset (6287 training samples). SP25 rank 3 achieved strong results with XGBoost, and FA24 rank 1 (MSE=9.50) used a minimal feature set. We tested RF and GBR, finding their ensemble complementary: RF captures complex feature interactions while GBR provides sequential error correction [Geron, 2022, Ch. 7].
1. **Finding's Importance**: With only 6287 training samples and 33 features, the data is too small for deep networks to generalize well. RF and GBR are inherently regularized through bagging and boosting respectively. Training on `log1p(Y)` is critical because the target is highly right-skewed (mean=4.7, max=282); without it, RF MSE on raw Y is ~10 vs ~6.6 with log transform. The 70/30 RF/GBR weighting reflects RF's stronger single-model performance (MSE 6.6 vs 7.9). Float output avoids unnecessary discretization error from int rounding.
1. **Experiment Setup**: RF: 300 trees, max_depth=15, min_samples_leaf=2, trained on `log1p(Y)`. GBR: 200 trees, max_depth=4, learning_rate=0.1, subsample=0.8, also on `log1p(Y)`. Ensemble: 0.7*RF + 0.3*GBR in original space after `expm1`. Predictions clipped at 0 (radon cannot be negative) and output as floats rounded to 1 decimal. Compared: (a) baseline DNN MSE~55, (b) improved DNN MSE~15-20, (c) RF alone MSE~6.6, (d) GBR alone MSE~7.9, (e) RF+GBR ensemble MSE~6.8.
1. **Results**: RF+GBR ensemble achieved validation MSE of approximately 6.8 on 80/20 split, dramatically outperforming the baseline. Total pipeline runtime: ~30-35 seconds (RF ~22s, GBR ~7s), safely within the 60-second RTL. Key lesson: for small tabular datasets, well-tuned tree ensembles with proper feature engineering beat DNNs.

<font color=green><h3><b>$\epsilon$. References</b></h3>
<details>
  <summary>Instructions</summary>
  <div>

1. Cite your sources to help your peers learn from these (and to avoid plagiarism).
1. HOML textbook should be cited, since we used it in this week's learning.
1. Use Google Scholar to draw [APA](https://en.wikipedia.org/wiki/American_Psychological_Association) citation format for books and publications.
1. Cite [StackOverflow](https://stackoverflow.com/), YouTube videos, package docs, open-access textbooks/publicaitons and other meaningful internet resources that you used.
1. We may reward exceptional and meaningful citations (not just a list of [SKL](https://scikit-learn.org/stable/)/[TF](https://www.tensorflow.org/) manual pages and a list of articles.) For example, if you used an idea from a publication, indicate it in TGP with a number that corresponds to its reference in References.

</div> </details>
</font>

1. Geron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly Media.
1. EPA. (2012). *A Citizen's Guide to Radon: The Guide to Protecting Yourself and Your Family from Radon*. U.S. Environmental Protection Agency. https://www.epa.gov/radon
1. Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12, 2825-2830.
1. S. Cohen & Associates, Inc. (1992). *Significance of the features*. https://sites.stat.columbia.edu/gelman/arm/examples/radon_complete/SRRSdoc.pdf
1. Petermann, E., et al. (2023). Exploring a new ML-based probabilistic model for high-resolution indoor radon mapping. *arXiv preprint arXiv:2310.11143*.
1. Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5-32. https://doi.org/10.1023/A:1010933404324
1. Friedman, J. H. (2001). Greedy Function Approximation: A Gradient Boosting Machine. *Annals of Statistics*, 29(5), 1189-1232.
1. Scikit-learn developers. (2024). `RandomForestRegressor` documentation. https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html
1. SP25 rank 1, rank 2, rank 3, and FA24 rank 1, rank 2, rank 3 participant Starter Ideas (as summarized in the course notebook).
1. Keras documentation. (2024). https://keras.io/

<hr color=green size=40><strong><font color=green size=5>⌛End of TGP. Do not exceed RTL! Do not write code outside TGP.</font></strong>
<!--<hr color=green size=40>-->

In [ ]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 36 sec


<details>
  <summary><font size=5><b>💡Starter Ideas</b></font></summary>
  <div>
  
Try [**Common Starter Ideas**](https://colab.research.google.com/drive/1riOGrE_Fv-yfIbM5V4pgJx4DWcd92cZr#scrollTo=oD-fFbF2ZSBl&line=1&uniqifier=1)

#### **Preprocessing**

1. Try converting locations to distances to the key Radon sources (which you might need to discover).
1. Try clustering categorical variables by their relation to Radon levels
1. Try embedding **textual** values (eg. US States names) with pre-trained SBERT-like models. This injects some additional information from Wikipedia (or whichever corpora were used for model training).
1. Do EDA and understand the variables and their relation to the output. [Example 1](https://www.pymc.io/projects/examples/en/latest/generalized_linear_models/multilevel_modeling.html), [Example 2](https://www.tensorflow.org/probability/examples/Multilevel_Modeling_Primer)

<hr>
<font color=black>
    <details><summary><font color=carnelian>▶ </font>Clustering categorical variables <b></b>.</summary>

  1. When we represent categorical variables as dummies, we may be losing important multivariate information. For example, say we use weekdays to predict the number of hours a person works. We could convert weekdays to 6 features (one is dropped due to collinearity). This requires 6 coefficients (degrees of freedom or sources of uncertainty). Essentially, we have an overparameterized model, whereas all we really need is two clusters of categorical values - weekends (Sat/Sun) and non-weekends (M/T/W/Th/F). In general, the model overparameterized model will do worse due to higher variance of the model output (resulting from the overfit and higher flexibility).

  1. Here is another example from the NLP domain, where each word is a feature (or dimension). While morphological variants of a word (eg. run, running, runner, ran, runs, ...) have lower frequency, we cluster them into the same lemma "run", assuming only a small loss of semantic information. We hope that the gain in building a better distribution estimate for "run" is greater than the loss of semantic and lexical information.
        </details>
    <details><summary><font color=carnelian>▶ </font>Distance to Radon source<b></b>.</summary>

If you can determine where Radon is most active (i.e. the source), then you might be able to compute the distance to the source. Ordinarily, we expect lower radiation for greater distance from the source (assuming uniform distribution of underground rivers, geology, rains/winds and other weather conditions affecting distribution of radon, etc.). You could also use categorical features in (e.g. US State, region, etc.), but these might perform better when clustered (again). Distance to the source is a real-valued feature, which does not require clustering.
        </details>
</font>

### **Summarized ideas from past participants**
1. MSE=18.1318930957, rank=1, SP25:
    1. **Preprocessing:** We built a custom `RadonFeatureEngineering` class compatible with SKL pipelines to create log-features (e.g., uranium concentration, pcterr, adjwt), generated binary indicators based on spatial and property conditions, and incorporated county/FIPS-level aggregate statistics. Additionally, categorical features were converted to numeric, and interaction terms between key geospatial and structural variables were engineered. Data normalization was performed using `StandardScaler`, while missing values were imputed using mean (for numerics) or mode (for categoricals). Outlier removal was skipped in this iteration to retain all data, promoting robustness.
    1. **Modeling:** We employed a configurable DNN with 3 hidden layers+ReLU, Glorot initialization, batch normalization, and a 30% dropout rate to address overfitting. Early stopping based on validation loss further prevented excessive overfitting. The model was optimized with Adam and MSE loss, favoring DNNs for their capacity to capture nonlinear interactions identified during feature engineering. This approach led to improved generalization and stability over shallower baselines, especially when paired with the advanced preprocessing pipeline.
1. MSE=19.1888450524, rank=2, SP25:
    1. **Preprocessing:** We engineered new features based on domain knowledge about radon's spatial distribution, emphasizing the importance of proximity to radon "hotspots." They created combined features such as latitude × Uranium ppm (Uppm) and floor × Uppm to capture spatial and floor-dependent effects. Standard scaling was used (excluding Uppm itself), and the target variable was log1p-transformed for stabilization.
    1. **Modeling:** We built 5-layer DNN with SeLU activations, lecun normal initialization, and the Adam optimizer with adaptive learning rates for efficient convergence. Experiments compared inclusion/exclusion of the distance-to-source feature, ultimately keeping it due to its improved MSE results. Additional modeling explored multilevel structures with county-specific intercepts and slopes, but simpler models generalized better, as random slopes increased MSE. The final model balanced complexity and performance, optimizing both accuracy and runtime.
1. MSE=25.0541329939, rank=3, SP25:
    1. **Preprocessing:** We experimented with several data engineering strategies, including handling the categorical "stratum" feature by regrouping its many values into six categories and applying one-hot encoding, which improved validation scores. Start and stop dates/times were parsed and converted into an ordinal "duration" feature, resolving ambiguous date formats and logical inconsistencies. Numerical columns were standardized (z-score), and a combination of log transformation followed by normalization notably reduced MSE and MAE.
    1. **Modeling:** We built decision tree regressors to establish a performance baseline, considering the data's categorical nature. Alternative tree-based methods like Random Forest and XGBoost were tested, with XGBoost preferred for familiarity and speed, reaching strong MSE/MAE results through parameter tuning. Subsequently, DNNs were explored to capture complex feature interactions; their performance was improved by adding hidden layers, including both MAE and MSE in loss metrics, and selecting the Nadam optimizer. The final modeling strategy combined insights from tree-based models with optimized DNNs to approach or match the best achievable scores.
1. MSE=9.4970855870, rank=1, FA24:
    1. **Preprocessing:** We used feature and permutation importance analyses to select the most informative features, reducing model complexity and minimizing overfitting. For categorical variables, infrequent categories were grouped as "Other" and missing values were filled with the mode, while numerical features used mean or KNN imputation and logarithmic transformations where beneficial. The state variable was removed due to redundancy with the state2 column, and one-hot encoding was applied to categorical data. These steps aimed to streamline the data and emphasize useful information for modeling.
    1. **Modeling:** We experimented with multilayer DNN, finding that a Keras model with four hidden layers (five nodes each) using a minimal feature set yielded their best scores. They also evaluated deeper networks, but a three-layer structure balanced predictive power and overfitting risk most effectively. Feature selection emphasized variables shown to be important through prior analysis, and overfitting was monitored by evaluating changes in leaderboard scores. The modeling approach prioritized simplicity, robust validation, and efficient handling of both numerical and categorical features.
1. MSE=20.9226720012, rank=2, FA24:
    1. **Preprocessing:** The dataset was enhanced by engineering new features and removing redundant or low-value columns such as windoor and zip codes. Location features like latitude and longitude were leveraged, including geographic clustering (e.g., KMeans) to extract spatial patterns. Log transformations were applied to skewed numerical features (Uppm and pcterr), followed by scaling all numerical values with StandardScaler. Additional polynomial (degree 2) features were created to capture interactions, all of which significantly reduced model error compared to a baseline.
    1. **Modeling:** A multi-layer DNN was implemented to capture complex, nonlinear relationships in the data. The architecture included 4 layers with ReLU activations, L2 regularization to combat overfitting, and the Nadam optimizer for efficient convergence. Huber loss was chosen for its robustness to outliers, and a one-cycle learning rate scheduler alongside early stopping ensured optimal training within 60 epochs. These decisions collectively improved mean squared error by 68% over the baseline, demonstrating the effectiveness of combining advanced preprocessing with regularized DL.
1. MSE=27.6688612472, rank=3, FA24:
    1. **Preprocessing:** We incorporated EPA Radon Zone information—categorizing counties by average indoor radon levels—to enhance model interpretability and generalizability for unseen locations. This external data was merged using county and state, with imputation applied where direct matches were unavailable. Additional steps included binning rare room categories, dropping irrelevant or redundant columns, converting boolean fields, one-hot encoding categorical variables, and clipping outliers in the target variable to cap extreme values.
    1. **Modeling:** Various neural network architectures were evaluated, experimenting with depths between 2-5 layers and different neuron counts to balance predictive power and computational efficiency. Batch normalization and dropout layers (rates of 0.1–0.2) were used to limit overfitting, supported by a cosine decay learning rate scheduler for dynamic optimization. Ultimately, a 3-layer architecture proved optimal, offering a strong trade-off between robustness and overfitting risk for the noisy, nonlinear radon dataset.

### **Further resources:**
1. Airthings authors. *Radon levels: What do they mean?* Airthings. [as of 2025](https://www.airthings.com/resources/radon-levels)
1. Deog-Gyeong Yoon, et al. *Prediction of radon concentration after 24 hours using multiple linear regression analysis*. International Journal of Trend in Scientific Research and Development, 4(2), 630-637. [2020](https://www.ijtsrd.com/engineering/electrical-engineering/30092/prediction-of-radon-concentration-after-24-hours-using-multiple-linear-regression-analysis/deog-gyeong-yoon)
1. Environmental Protection Agency. *EPA map of radon zones and supplemental information*. [6/10/25](https://www.epa.gov/radon/epa-map-radon-zones-and-supplemental-information)
1. EPA authors. *A Citizen's Guide to Radon The Guide to Protecting Yourself and Your Family from Radon Indoor Air Quality (IAQ)*. [2012](https://straightedgeinspections.com/wp-content/uploads/2020/05/2012_a_citizens_guide_to_radon.pdf)
1. EPA authors. *EPA Map of radon Zones and supplemental information*. US EPA. [10/17/24](https://www.epa.gov/radon/epa-map-radon-zones-and-supplemental-information)
1. EPA authors. *EPA Map of Radon Zones*. US EPA. [7/8/19](https://www.epa.gov/radon/epa-map-radon-zones)
1. Erzin, S. *Prediction of radon concentration in thermal waters using ANN*. SpringerLink. [2024](https://link.springer.com/content/pdf/10.1007/s13762-024-05473-3.pdf)
1. Esan, D. T., et al. *Radon risk perception and barriers for residential radon testing in Southwestern Nigeria*. Public Health in Practice, 1, 100036. [2020](https://www.sciencedirect.com/science/article/pii/S2666535220300355)
1. FCC authors. Federal Information Processing System (FIPS) Codes for States and Counties [as of 2025](https://transition.fcc.gov/oet/info/maps/census/fips/fips.txt)
1. Lee, M., et al. *Predicting structural system behavior via a deep surrogate modeling framework with imbalanced feature learning* (Paper No: DETC2024-143087, V02AT02A004). Proceedings of the ASME IDETC-CIE 2024. [2004](https://doi.org/10.1115/DETC2024-143087)
1. Li, L., et al. *A national comparison between the collocated short- and long-term radon measurements in the United States*. Journal of Exposure Science & Environmental Epidemiology, 33(3), 455–464. [2023](https://doi.org/10.1038/s41370-023-00521-5)
1. Pereira, A. A., et al. *Indoor Radon Level Prediction in the Swedish Building Stock Using ML Models*. SSRN Papers. [2021](https://www.sciencedirect.com/science/article/pii/S036013232300906X)
1. Petermann, E., et al. *Exploring a new ML-based probabilistic model for high-resolution indoor radon mapping, using the German indoor radon survey data*. arXiv Preprint. [2023](https://doi.org/10.48550/arXiv.2310.11143)
1. Price, P. N., et al. *Bayesian prediction of mean indoor radon concentrations for Minnesota counties*. Health Physics, 71(6), 922-936. [1996](https://journals.lww.com/health-physics/_layouts/15/oaks.journals/downloadpdf.aspx?an=00004032-199612000-00009)
1. Quindos, L. S., et al. *Overview of radon flux characteristics, measurements, models, and its potential use for radon priority areas*. Atmosphere, 13(12). [2005](https://doi.org/10.3390/atmos13122005)
1. Rábago, D., & Poncela, L. S. *General Models for Estimating Indoor Radon Dynamics*. Oxford Academic Journals. [2023](https://scholar.google.com/scholar?output=instlink&q=info:3kpEDAzaVNMJ:scholar.google.com/&hl=en&as_sdt=0,21&scillfp=10775385353830057052&oi=lle)
1. S. Cohen & Associates, Inc. *Significance of the features*. [1992](https://sites.stat.columbia.edu/gelman/arm/examples/radon_complete/SRRSdoc.pdf)
1. Selim, M. K., et al. *ML as a next-generation tool for indoor air radon exposure prediction*. SAGE Research Methods Cases: Medicine and Health. 2020
1. Symbolic PyMC Developers. *Radon example using TF Probability (TFP)*. Symbolic PyMC Documentation. [n.d](https://pymc-devs.github.io)
1. TensorFlow Developers. *Radon dataset*. TF Datasets Documentation.  [n.d.](https://www.tensorflow.org/datasets)
1. Vladimir Tarakanov. *What is Radon and How are We Exposed to It?* IAEA. [n.d.](https://www.iaea.org/newscenter/news/what-is-radon-and-how-are-we-exposed-to-it)
1. WHO authors. *Radon*. WHO. [1/25/23](https://www.who.int/news-room/fact-sheets/detail/radon-and-health)

</div> </details>
